# 1. wrap_tool_call 处理审批逻辑（对齐官方 HumanInTheLoopMiddleware 协议）

`wrap_tool_call` 是 `ToolNode` 的干预钩子：**每个工具调用执行前**都会经过它——审批逻辑正好装在这里。

- 钩子签名：`(request: ToolCallRequest, execute) -> ToolMessage | ...`；`execute(request)` 执行工具，`request.override(tool_call=...)` 可换参数
- 审批 = 钩子内调 `interrupt()` 挂起，恢复值按**官方四决策协议**处理（与 `HumanInTheLoopMiddleware` 一致）：

| 决策 | resume 值 | 语义 |
|---|---|---|
| approve | `{"type": "approve"}` | 执行原工具 |
| edit | `{"type": "edit", "edited_action": {...}}` | 保留原 `tool_call_id`，换参数执行 |
| reject | `{"type": "reject"}` | 不执行，回填 `status="error"` 的 ToolMessage（默认文案劝阻模型重试） |
| respond | `{"type": "respond", "message": ...}` | 人代替工具回答，回填 `status="success"` 的 ToolMessage |

In [ ]:
from typing import Literal

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphInterrupt
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt.tool_node import ToolCallRequest  # 注意：不在 prebuilt 顶层导出
from langgraph.types import Command, interrupt
from rich import print

load_dotenv(override=True)

model = init_chat_model(
    model_provider="deepseek",
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)


# 定义工具
@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    return {
        "科技": "最新科技新闻：AI 技术正在快速发展。",
        "体育": "最新体育新闻：中国队取得了胜利。",
        "娱乐": "最新娱乐新闻：歌手发布了新专辑。",
    }[topic]


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


# ===== 审批策略：对齐官方 HumanInTheLoopMiddleware =====
# 每种工具允许用户做哪些决策；不在表中的工具不需要审批，直接放行
REVIEW_TOOLS: dict[str, list[str]] = {
    "get_weather": ["approve", "edit", "reject", "respond"],
}


def review_tool_call(request: ToolCallRequest, execute) -> ToolMessage | object:
    """工具审批钩子：每个工具调用执行前都会经过这里"""
    tc = request.tool_call
    allowed = REVIEW_TOOLS.get(tc["name"])
    if allowed is None:
        return execute(request)  # 不在审批名单 → 直接放行

    decision = interrupt({  # ===== 挂起，等用户决策 =====
        "action_request": {"action": tc["name"], "args": tc["args"]},
        "allowed_decisions": allowed,
        "tool_call_id": tc["id"],
    })

    if decision["type"] == "approve" and "approve" in allowed:
        return execute(request)  # 批准：执行原工具

    if decision["type"] == "edit" and "edit" in allowed:  # 编辑：保留 id，换参数
        edited = decision["edited_action"]
        new_tc = {**tc, "name": edited["name"], "args": edited["args"]}
        return execute(request.override(tool_call=new_tc))

    if decision["type"] == "reject" and "reject" in allowed:  # 拒绝：不执行，回填错误
        content = decision.get("message") or (
            f"User rejected the tool call for `{tc['name']}` with id {tc['id']}. "
            "The tool was not executed. Do not retry this tool call unless the user "
            "explicitly requests it."
        )
        return ToolMessage(content=content, name=tc["name"],
                           tool_call_id=tc["id"], status="error")

    if decision["type"] == "respond" and "respond" in allowed:  # 代答：人代替工具回答
        return ToolMessage(content=decision["message"], name=tc["name"],
                           tool_call_id=tc["id"], status="success")

    raise ValueError(f"不支持的决策: {decision}，允许的决策: {allowed}")


def handle_errors(e: Exception) -> str:
    # 坑：handle_tool_errors=True 会把 GraphInterrupt 当工具异常吞掉，审批失效
    # 中断是控制流信号，必须放行
    if isinstance(e, GraphInterrupt):
        raise e
    return f"工具执行出错: {e}"


# 工具节点：钩子接管审批，错误按 handler 策略处理
tool_node = ToolNode(tools, wrap_tool_call=review_tool_call,
                     handle_tool_errors=handle_errors)


# 定义状态
class State(MessagesState):
    pass


# 模型节点
def llm_node(state: State) -> State:
    ai_msg = model_with_tools.invoke(state["messages"])
    return {"messages": [ai_msg]}


def router(state: State) -> Literal["tool_node", END]:
    return "tool_node" if state["messages"][-1].tool_calls else END


builder = StateGraph(state_schema=State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", router, [END, "tool_node"])
builder.add_edge("tool_node", "llm_node")

# 坑：恢复必须挂 checkpointer，否则 Command(resume=...) 直接 RuntimeError
graph = builder.compile(checkpointer=InMemorySaver())

In [ ]:
from IPython.display import display

print(display(graph))

# 2. 审批循环：invoke → 暂停 → resume

一次带审批的调用分两步：

1. 第一次 `invoke`：模型发起工具调用 → 钩子内 `interrupt()` 挂起 → 结果里带 `__interrupt__`（审批请求 payload）；
2. 第二次 `invoke(Command(resume=决策值), config)`：决策值注入 `interrupt()` 的返回值，钩子按四决策协议处理后继续执行。

> 注意：interrupt 恢复时**工具节点的代码会从头重跑**，免审批工具的"放行"和已做的决策都会按缓存值快速重放——副作用必须发生在 `execute()`（拿到许可）之后。

In [ ]:
def run_scenario(name: str, question: str, resume_value):
    config = {"configurable": {"thread_id": name}}
    print(f"\n===== 场景 {name} =====")
    res = graph.invoke({"messages": [HumanMessage(content=question)]}, config)
    print("审批请求 ->", res["__interrupt__"][0].value)
    res = graph.invoke(Command(resume=resume_value), config)
    for m in res["messages"][2:]:
        print(type(m).__name__, "|", m.content if m.content else m.tool_calls)


# ① 批准：工具按原参数执行
run_scenario("批准", "查一下上海的天气", {"type": "approve"})
# ② 编辑：参数被换成北京，模型会如实发现"我要上海，返回的是北京"
run_scenario("编辑", "查一下上海的天气",
             {"type": "edit", "edited_action": {"name": "get_weather", "args": {"city": "北京"}}})
# ③ 拒绝：工具未执行，官方默认文案劝阻模型重试
run_scenario("拒绝", "查一下上海的天气", {"type": "reject"})
# ④ 代答：人不执行工具，直接以工具身份回答
run_scenario("代答", "查一下上海的天气",
             {"type": "respond", "message": "天气服务正在维护，请直接告诉用户：暂时无法查询"})

# 3. 实测踩到的坑

1. **`handle_tool_errors=True` 会吞掉 `GraphInterrupt`**：中断被包成 `Error: GraphInterrupt(...)` 的 ToolMessage 回给模型，审批直接失效。必须用自定义 handler 把 `GraphInterrupt` `raise` 放行（见 `handle_errors`）。
2. **恢复必须挂 checkpointer**：`Command(resume=...)` 没有 checkpointer 直接 `RuntimeError: Cannot use Command(resume=...) without checkpointer`。
3. 官方 reject 默认文案能明显降低模型盲目重试，但不保证——拒绝场景模型可能仍说"我可以再次尝试"，只是没有真的再调工具。

> 知识点详解见 [CheatSheet.md](CheatSheet.md) 第七节。